In [ ]:
#import process_RV_gradient
"""
Image Quality Metrics: SNR and CNR for cardiac NIfTI images.

Add this function to your existing script, or run standalone.

Label assumptions (adjust if your segmentation uses different label IDs):
  - RV blood pool labels: [4, 5, 22, 24]
  - Myocardium labels:    [2]  # LV myocardium — change to your actual myo label(s)
"""

import nibabel as nib
from nilearn.image import resample_to_img
import numpy as np
import os

def compute_snr_cnr(
    dcmfile,
    maskfile,
    rv_labels=[4], # , 5, 22, 24],
    myo_labels=[2,3],
):
    """
    Compute SNR in the RV blood pool and CNR between RV blood pool and myocardium.

    SNR = mean(RV) / std(RV)
    CNR = |mean(RV) - mean(Myo)| / std(background)

    Parameters
    ----------
    dcmfile : str
        Path to the DCM NIfTI image (e.g. dcm-0.nii).
    maskfile : str
        Path to the segmentation mask (e.g. all_labels_mask.nii).
    rv_labels : list of int
        Label values corresponding to the RV blood pool.
    myo_labels : list of int
        Label values corresponding to the myocardium.

    Returns
    -------
    dict with keys: snr_rv, cnr_rv_myo, mean_rv, std_rv, mean_myo, std_myo, std_background
    """

    # --- Load and resample (same logic as your existing code) ---
    dcm_img = nib.load(dcmfile)
    dcm_img.set_sform(dcm_img.get_qform(), code=1)

    mask_img = nib.load(maskfile)
    mask_img.set_sform(mask_img.get_qform(), code=1)

    resampled_mask_img = resample_to_img(
        mask_img, dcm_img, interpolation="nearest",
        force_resample=True, copy_header=True,
    )

    dcm_data = dcm_img.get_fdata()
    mask_data = resampled_mask_img.get_fdata()

    # --- Build binary masks ---
    rv_mask = np.isin(mask_data, rv_labels)
    myo_mask = np.isin(mask_data, myo_labels)
    # Background = everything that is NOT any label (label == 0)
    bg_mask = mask_data == 0

    # --- Extract voxel intensities ---
    rv_vals = dcm_data[rv_mask]
    myo_vals = dcm_data[myo_mask]
    bg_vals = dcm_data[bg_mask]

    # --- Sanity checks ---
    if rv_vals.size == 0:
        raise ValueError(
            f"No voxels found for RV blood pool labels {rv_labels}. "
            "Check that your label IDs match the segmentation mask."
        )
    if myo_vals.size == 0:
        raise ValueError(
            f"No voxels found for myocardium labels {myo_labels}. "
            "Check that your label IDs match the segmentation mask."
        )

    mean_rv = np.mean(rv_vals)
    std_rv = np.std(rv_vals)
    mean_myo = np.mean(myo_vals)
    std_myo = np.std(myo_vals)
    std_bg = np.std(bg_vals)

    # --- SNR: signal-to-noise in RV blood pool ---
    snr_rv = mean_rv / std_rv if std_rv > 0 else float("inf")

    # --- CNR: contrast-to-noise between RV blood pool and myocardium ---
    #     Using background noise in the denominator (standard definition)
    cnr_rv_myo = abs(mean_rv - mean_myo) / std_bg if std_bg > 0 else float("inf")

    results = {
        "snr_rv": snr_rv,
        "cnr_rv_myo": cnr_rv_myo,
        "mean_rv": mean_rv,
        "std_rv": std_rv,
        "mean_myo": mean_myo,
        "std_myo": std_myo,
        "std_background": std_bg,
        "n_voxels_rv": rv_vals.size,
        "n_voxels_myo": myo_vals.size,
    }

    return results

import os
# ── Standalone usage / notebook cell ─────────────────────────────────────────
if __name__ == "__main__":
    for case in range(1, 14):
        casepath=f"/data_fast/Work/MS_HF_cohort/data/{case:02d}/"
        dcmfile = f"{casepath}0_dcm/dcm-0.nii"
        maskfile = f"{casepath}1_seg/all_labels_mask.nii"
        # Check dcmfile and maskfile paths before running

        if not os.path.exists(dcmfile):
            print(f"Error: DCM file does not exist: {dcmfile}")
            continue
        if not os.path.exists(maskfile):
            print(f"Error: Mask file does not exist: {maskfile}")
            continue
        print (f"Processing case {case:02d}...")
        run_metrics(dcmfile, maskfile)

def run_metrics(dcmfile, maskfile):
    # ⚠️  UPDATE myo_labels to match YOUR segmentation convention
    metrics = compute_snr_cnr(
        dcmfile,
        maskfile,
        rv_labels=[4], #, 5, 22, 24],
        myo_labels=[2,3],          # <-- change if needed
    )

    #print("=" * 50)
    print("  Image Quality Metrics ")
    #print("=" * 50)
    # print(f"  RV blood pool voxels : {metrics['n_voxels_rv']}")
    # print(f"  Myocardium voxels   : {metrics['n_voxels_myo']}")
    # print(f"  Mean RV intensity   : {metrics['mean_rv']:.2f}")
    # print(f"  Std  RV intensity   : {metrics['std_rv']:.2f}")
    # print(f"  Mean Myo intensity  : {metrics['mean_myo']:.2f}")
    # print(f"  Std  Myo intensity  : {metrics['std_myo']:.2f}")
    # print(f"  Std  Background     : {metrics['std_background']:.2f}")
    print("-" * 50)
    print(f"  SNR (RV blood pool)        : {metrics['snr_rv']:.3f}")
    print(f"  CNR (RV vs Myocardium)     : {metrics['cnr_rv_myo']:.3f}")
    print("=" * 50)

Processing case 01...


  Image Quality Metrics 
--------------------------------------------------
  SNR (RV blood pool)        : 1.909
  CNR (RV vs Myocardium)     : 0.294
Processing case 02...
